# pptk Interactive 3D Viewer in Jupyter

This notebook demonstrates the inline Three.js point cloud visualization that renders directly in Jupyter cells.
Drag to orbit, scroll to zoom, right-click-drag to pan.

In [1]:
import numpy as np
import sys, os

# If pptk is not installed, add the repo root to sys.path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from pptk.viewer.viewer import viewer

## Helper: create a viewer without spawning the native Qt process

Since the Qt viewer binary may not be available in a headless environment (e.g. Binder, Colab),
we create viewer instances directly and populate their internal state.

In [2]:
def make_inline_viewer(positions, *attrs):
    """Create a viewer object for inline display without the native Qt process."""
    v = viewer.__new__(viewer)
    v._portNumber = 0
    v._process = None
    v._offset = np.zeros(3)
    v._positions = np.asarray(positions, dtype=np.float64).reshape(-1, 3)
    v._attr = attrs
    return v

## 1. Basic random point cloud

1000 random points in a unit cube, displayed with a default blue color.

In [3]:
xyz = np.random.rand(1000, 3)
v = make_inline_viewer(xyz)
v

## 2. RGB color attributes

Points colored by their (x, y, z) position mapped to (R, G, B).

In [ ]:
xyz = np.random.rand(5000, 3)
rgb = xyz.copy().astype(np.float32)  # color = position
v = make_inline_viewer(xyz, rgb)
v

## 3. Scalar attribute with jet colormap

Points colored by height (z-coordinate) using the jet colormap.

In [ ]:
# Sphere point cloud
n = 10000
theta = np.random.uniform(0, 2 * np.pi, n)
phi = np.arccos(np.random.uniform(-1, 1, n))
r = np.random.uniform(0.8, 1.0, n)
xyz = np.column_stack([
    r * np.sin(phi) * np.cos(theta),
    r * np.sin(phi) * np.sin(theta),
    r * np.cos(phi),
])
height = xyz[:, 2].astype(np.float32)
v = make_inline_viewer(xyz, height)
v

## 4. Large point cloud (subsampling)

When more than 200k points are provided, the viewer automatically subsamples for browser performance.
The info overlay shows the original count and the subsampled count.

In [ ]:
xyz_large = np.random.rand(500_000, 3) * 10
scalar = np.linalg.norm(xyz_large - 5.0, axis=1).astype(np.float32)
v = make_inline_viewer(xyz_large, scalar)
v

## 5. Structured shapes

A torus point cloud colored by angle, showing the jet colormap on a recognizable shape.

In [ ]:
n = 20000
u = np.random.uniform(0, 2 * np.pi, n)
v_angle = np.random.uniform(0, 2 * np.pi, n)
R, r_tube = 2.0, 0.6
x = (R + r_tube * np.cos(v_angle)) * np.cos(u)
y = (R + r_tube * np.cos(v_angle)) * np.sin(u)
z = r_tube * np.sin(v_angle)
xyz_torus = np.column_stack([x, y, z])

v = make_inline_viewer(xyz_torus, u.astype(np.float32))
v

## Using with the full pptk viewer

If you have the native Qt viewer built and installed, you can use the normal `pptk.viewer()` API.
The `_repr_html_()` method is called automatically when the viewer object is the last expression in a cell:

```python
import pptk
v = pptk.viewer(xyz, rgb)
v  # displays interactive 3D view inline
```

The native viewer window will also open as usual for full-featured interaction.